In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Library Model
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler # LR Wajib pake Scaler!
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

# 1. Definisi Rumus Evaluasi
def smape(y_true, y_pred):
    """sMAPE: Error persen yang aman dari angka 0."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator > 0
    if not np.any(mask): return 0.0
    return np.mean(numerator[mask] / denominator[mask]) * 100

def wape(y_true, y_pred):
    """WAPE: Metric Bisnis. Total Error dibagi Total Omzet."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true))
    if denominator == 0: return np.inf
    return (numerator / denominator) * 100

sns.set_style("whitegrid")
plt.rc('figure', figsize=(12, 6))

try:
    # 2. Load & Clean Data
    print("Loading sales_data.csv untuk Linear Regression (REVISI)...")
    df = pd.read_csv("sales_data.csv")
    df['week_start'] = pd.to_datetime(df['week_start'])
    df = df.dropna(subset=['total_sales', 'seller_id', 'product_category_name'])

    # SORTING WAKTU (Wajib!)
    df = df.sort_values(by=['week_start', 'seller_id']).reset_index(drop=True)

    # 3. OUTLIER CLEANING (Biar Adil sama RF)
    # Kita buang data sales yang ga ngotak tingginya biar Linear Regression ga "meledak"
    Q1 = df['total_sales'].quantile(0.25)
    Q3 = df['total_sales'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 3.0 * IQR # Batas atas ekstrem

    print(f"\n[INFO CLEANING]")
    print(f"Membuang data sales di atas: {upper_bound:.2f}")

    initial_rows = len(df)
    df = df[df['total_sales'] <= upper_bound].reset_index(drop=True)
    print(f"Data dibuang: {initial_rows - len(df)} baris")
    print(f"Sisa data bersih: {len(df)} baris")

    # 4. Feature Engineering (Anti-Leakage)
    # Expanding Mean (Shift 1): Rata-rata masa lalu murni
    df['seller_avg_sales'] = df.groupby('seller_id')['total_sales'].transform(lambda x: x.shift(1).expanding().mean()).fillna(0)
    df['seller_order_count'] = df.groupby('seller_id')['total_orders'].transform(lambda x: x.shift(1).expanding().sum()).fillna(0)
    df['seller_weeks_active'] = df.groupby('seller_id').cumcount()

    df['month'] = df['week_start'].dt.month
    df['week_of_year'] = df['week_start'].dt.isocalendar().week.astype(int)

    df = df[df['seller_weeks_active'] > 0].reset_index(drop=True)
    df['total_sales_log'] = np.log1p(df['total_sales']) # Target Log

    # 5. Preparation & Training
    print("\nTraining Linear Regression (Versi Jujur)...")

    # DROP KOLOM BOCOR (Sama persis kayak RF)
    cols_to_drop = [
        'total_sales', 'total_sales_log',
        'week_start', 'seller_city', 'seller_id',
        'total_items', 'total_orders', 'avg_order_value' # <DIBUANG!
    ]

    X = df.drop(cols_to_drop, axis=1)
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = ['product_category_name', 'seller_state']
    y = df['total_sales_log']

    # Preprocessor
    # BEDANYA SAMA RF: Linear Regression WAJIB pake StandardScaler buat angka
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features), # Scaling Wajib!
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ], remainder='drop')

    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression()) # Model Linear
    ])

    # SPLIT TIME BASED
    split_index = int(len(df) * 0.8)
    X_train = X.iloc[:split_index]
    y_train = y.iloc[:split_index]
    X_test = X.iloc[split_index:]
    y_test = y.iloc[split_index:]

    print(f"Data Train Final: {len(X_train)} baris")

    model_pipeline.fit(X_train, y_train)
    print("Training Selesai.")

    # 6. Evaluasi
    log_predictions = model_pipeline.predict(X_test)

    # CLIPPING (PENTING BUAT LR)
    # Kita batasin prediksinya biar ga meledak pas di-inverse
    # Batas maks = nilai log tertinggi di training + buffer dikit
    max_log_train = y_train.max() + 0.5
    log_predictions = np.clip(log_predictions, a_min=0, a_max=max_log_train)

    # Inverse Log
    y_test_original = np.expm1(y_test)
    predictions_original = np.expm1(log_predictions)

    rmse = np.sqrt(mean_squared_error(y_test_original, predictions_original))
    r2 = r2_score(y_test_original, predictions_original)
    wape_val = wape(y_test_original, predictions_original)

    print(f"\nlinear reg. result")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2: {r2:.4f}")
    print(f"WAPE: {wape_val:.2f}%")

    # Visualisasi
    plt.figure(figsize=(10, 10))
    sns.scatterplot(x=y_test_original, y=predictions_original, alpha=0.3)
    limit_min = min(y_test_original.min(), predictions_original.min())
    limit_max = max(y_test_original.max(), predictions_original.max())
    plt.plot([limit_min, limit_max], [limit_min, limit_max], '--', color='red', linewidth=2, label='Prediksi Sempurna')
    plt.title('Linear Regression: Aktual vs Prediksi', fontsize=16)
    plt.xlabel('Sales Asli', fontsize=12)
    plt.ylabel('Prediksi', fontsize=12)
    plt.legend()
    plt.show()

except Exception as e:
    print(f"Error: {e}")